# Tiled attention and online softmax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/flash_attention.ipynb)

Self-contained CPU experiment. Install missing dependencies with `%pip install torch numpy scipy matplotlib`. Retain Colab's supplied PyTorch build. Tested on Python 3.12, PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development-machine GPU build.

In [ ]:
import math, time
import numpy as np
import torch
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
torch.set_num_threads(4)

## Dense attention

The dense reference materialises the query-key score matrix. This is the correctness reference for the tiled calculation.

In [ ]:
def dense_attention(q, k, v):
    return (q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])).softmax(-1) @ v

## Tiled attention

Each outer iteration fixes a query block. Inner key/value blocks update the running maximum, denominator and unnormalised output. The final division happens once. This follows the query-outer arrangement of FlashAttention-2, using ordinary PyTorch operations.

In [ ]:
def tiled_attention(q, k, v, block=64):
    """Unmasked exact attention; ordinary tensor ops, no custom fused kernel."""
    outputs = []
    for qi in q.split(block, dim=-2):
        shape = (*qi.shape[:-1], 1)
        m = torch.full(shape, -torch.inf, dtype=q.dtype, device=q.device)
        ell = torch.zeros(shape, dtype=q.dtype, device=q.device)
        acc = torch.zeros_like(qi)
        for kj, vj in zip(k.split(block, dim=-2), v.split(block, dim=-2)):
            scores = qi @ kj.transpose(-2, -1) / math.sqrt(q.shape[-1])
            new_m = torch.maximum(m, scores.max(-1, keepdim=True).values)
            alpha = (m - new_m).exp()
            p = (scores - new_m).exp()
            acc = alpha * acc + p @ vj
            ell = alpha * ell + p.sum(-1, keepdim=True)
            m = new_m
        outputs.append(acc / ell)
    return torch.cat(outputs, dim=-2)

## Flash experiment

The float64 check tests both values and gradients. Float32 checks run at every benchmark length. Forward timings use no_grad, two warmups, five synchronised measurements and incremental PyTorch allocation statistics on GPU. CPU memory is not measured.

In [ ]:
def flash_experiment(device='cpu'):
    start = time.perf_counter()
    torch.manual_seed(117)
    q, k, v = [torch.randn(1, 2, 19, 8, dtype=torch.float64, requires_grad=True) for _ in range(3)]
    a = dense_attention(q, k, v)
    b = tiled_attention(q, k, v, 7)
    probe = torch.randn_like(a)
    ga = torch.autograd.grad((a * probe).sum(), (q, k, v), retain_graph=True)
    gb = torch.autograd.grad((b * probe).sum(), (q, k, v))
    errors = {'output': float((a - b).detach().abs().max()), 'gradient': max((float((x - y).abs().max()) for x, y in zip(ga, gb)))}
    assert errors['output'] < 1e-12 and errors['gradient'] < 1e-12
    records = []
    device_errors = []
    for dev in dict.fromkeys(['cpu', device]):
        for n in [128, 256, 512, 1024, 2048]:
            inputs = [torch.randn(1, 2, n, 32, device=dev, dtype=torch.float32) for _ in range(3)]
            with torch.no_grad():
                expected = dense_attention(*inputs)
                for label, check in [('tiled', tiled_attention), ('sdpa_auto', F.scaled_dot_product_attention)]:
                    actual = check(*inputs)
                    err = float((actual - expected).abs().max())
                    assert torch.allclose(actual, expected, atol=2e-05, rtol=0.0002)
                    device_errors.append(dict(device=dev, n=n, method=label, max_abs=err))
                del expected, actual
            for name, fn in [('dense', dense_attention), ('tiled', tiled_attention), ('sdpa_auto', F.scaled_dot_product_attention)]:
                with torch.no_grad():
                    for _ in range(2):
                        fn(*inputs)
                    if dev == 'cuda':
                        torch.cuda.synchronize()
                        torch.cuda.reset_peak_memory_stats()
                        base = torch.cuda.memory_allocated()
                    ts = []
                    for _ in range(5):
                        t = time.perf_counter()
                        result = fn(*inputs)
                        if dev == 'cuda':
                            torch.cuda.synchronize()
                        ts.append((time.perf_counter() - t) * 1000)
                        del result
                    peak = torch.cuda.max_memory_allocated() - base if dev == 'cuda' else None
                records.append(dict(device=dev, n=n, method=name, median_ms=float(np.median(ts)), peak_extra_bytes=peak))
    backends = {}
    if device == 'cuda':
        from torch.nn.attention import sdpa_kernel, SDPBackend
        for name, be in [('FLASH_ATTENTION', SDPBackend.FLASH_ATTENTION), ('EFFICIENT_ATTENTION', SDPBackend.EFFICIENT_ATTENTION), ('MATH', SDPBackend.MATH)]:
            try:
                with sdpa_kernel(be):
                    F.scaled_dot_product_attention(*inputs)
                    torch.cuda.synchronize()
                backends[name] = 'supported for measured float32 shape'
            except Exception as e:
                backends[name] = str(e).splitlines()[0]
    rng = np.random.default_rng(117)
    scores = rng.normal(0, 5, 96)
    values = rng.normal(size=(96, 2))
    m = -np.inf
    ell = 0.0
    acc = np.zeros(2)
    trace = []
    for j in range(0, 96, 8):
        z = scores[j:j + 8]
        new = max(m, z.max())
        alpha = np.exp(m - new)
        p = np.exp(z - new)
        acc = alpha * acc + p @ values[j:j + 8]
        ell = alpha * ell + p.sum()
        m = new
        trace.append({'keys': j + 8, 'max': float(m), 'sum': float(ell), 'output': (acc / ell).tolist()})
    return dict(errors=errors, device_errors=device_errors, records=records, backends=backends, trace=trace, seconds=time.perf_counter() - start, torch=torch.__version__)

## Probe attention backends

Optional GPU diagnostic: force each dtype/backend combination, keep complete warnings, and inspect automatic-dispatch operator names. Merely passing an API call does not identify the backend.

In [ ]:
def probe_attention_backends():
    """Record complete dtype-specific backend failures and automatic dispatch."""
    import warnings
    from torch.nn.attention import sdpa_kernel, SDPBackend
    from torch.profiler import profile, ProfilerActivity
    torch.manual_seed(117)
    outcomes = []
    for dtype in [torch.float32, torch.float16, torch.bfloat16]:
        inputs = [torch.randn(1, 2, 2048, 32, dtype=dtype, device='cuda') for _ in range(3)]
        for label, backend in [('FLASH_ATTENTION', SDPBackend.FLASH_ATTENTION),
                               ('EFFICIENT_ATTENTION', SDPBackend.EFFICIENT_ATTENTION),
                               ('MATH', SDPBackend.MATH)]:
            row = dict(dtype=str(dtype), backend=label)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                try:
                    with torch.no_grad(), sdpa_kernel(backend):
                        output = F.scaled_dot_product_attention(*inputs)
                        torch.cuda.synchronize()
                    row['success'] = True
                    row['finite'] = bool(output.isfinite().all())
                    reference = dense_attention(*[x.float() for x in inputs])
                    row['max_abs_error_vs_float32'] = float((output.float()-reference).abs().max())
                except Exception as exc:
                    row['success'] = False
                    row['error'] = str(exc)
                row['warnings'] = [str(w.message) for w in caught]
            outcomes.append(row)
        with torch.no_grad(), profile(activities=[ProfilerActivity.CPU]) as prof:
            F.scaled_dot_product_attention(*inputs)
            torch.cuda.synchronize()
        outcomes.append(dict(dtype=str(dtype), backend='automatic',
            operators=[event.key for event in prof.key_averages() if 'attention' in event.key.lower()]))
    return outcomes

## Run and print the complete comparison

All measurements below are calculated in this notebook. Fixed seeds fix samples, while hardware and library changes can affect timing and floating-point details.

In [ ]:
result = flash_experiment('cpu')
def print_table(result, topic):
    """Print every experimental row, followed by the article's grouped means."""
    print('device length method median_ms extra_gpu_MiB')
    for row in result['records']:
        memory = None if row['peak_extra_bytes'] is None else row['peak_extra_bytes'] / 2 ** 20
        print(row['device'], row['n'], row['method'], round(row['median_ms'], 3), memory)
    print('Double precision checks:', result['errors'])
    print('All float32 checks:', result['device_errors'])
    print('Forced float32 backend outcomes:', result['backends'])
print_table(result, 'flash')

## Flash plots

The figure uses the newly computed result, with baselines and units labelled.

In [ ]:
def flash_plots(result):
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = {'dense': 'Dense attention', 'tiled': 'Python tiled', 'sdpa_auto': 'Automatic SDPA'}
    gpu = any(row['device'] == 'cuda' for row in result['records'])
    for method, label in labels.items():
        rows = [row for row in result['records'] if row['device'] == ('cuda' if gpu else 'cpu') and row['method'] == method]
        axes[0].plot([row['n'] for row in rows], [row['median_ms'] for row in rows], 'o-', label=label)
        if gpu:
            axes[1].plot([row['n'] for row in rows], [row['peak_extra_bytes']/2**20 for row in rows], 'o-', label=label)
    axes[0].set(xlabel='Sequence length', ylabel='Median forward latency (ms)', yscale='log', title='GPU' if gpu else 'CPU')
    if gpu:
        axes[1].set(xlabel='Sequence length', ylabel='Extra PyTorch GPU allocation (MiB)', yscale='log')
    else:
        sizes = np.array([128, 256, 512, 1024, 2048])
        axes[1].plot(sizes, 2*sizes*sizes*4/2**20, label='Dense score tensor')
        axes[1].plot(sizes, np.full_like(sizes, 2*64*64*4/2**20, dtype=float), label='One tiled score block')
        axes[1].set(xlabel='Sequence length', ylabel='Analytical score storage (MiB)')
    for axis in axes:
        axis.legend(fontsize=10)
    figure.tight_layout()
    return figure
flash_plots(result)
plt.show()

## Animated experiment

The synthetic Normal(0,25) scores illustrate accumulator updates; dotted lines mark changes in the running maximum.

In [ ]:
def progress_animation(result, topic):
    figure, axes = plt.subplots(figsize=(8, 4.5))
    frames = list(range(15)) + [14] * 6

    def update(frame):
        fraction = (frame + 1) / 15
        axis = axes
        axis.clear()
        count = max(1, int(fraction * len(result['trace'])))
        rows = result['trace'][:count]
        for coordinate in [0, 1]:
            axis.plot([row['keys'] for row in rows], [row['output'][coordinate] for row in rows], 'o-', label=f'Output coordinate {coordinate + 1}')
        for i, row in enumerate(rows):
            if i == 0 or row['max'] > rows[i - 1]['max']:
                axis.axvline(row['keys'], color='grey', alpha=0.25, linestyle=':')
        values = [v for row in result['trace'] for v in row['output']]
        axis.set(xlim=(0, 100), ylim=(min(values) - 0.2, max(values) + 0.2), xlabel='Keys processed', ylabel='Running attention output', title='Synthetic scores: Normal(0, 5²); dotted lines: new maximum')
        axis.legend(fontsize=9)
        figure.tight_layout()
        return []
    return FuncAnimation(figure, update, frames=frames, interval=333, blit=False)
from IPython.display import Image, display
animation = progress_animation(result, 'flash')
animation.save('flash_progress.gif', writer=PillowWriter(fps=3))
plt.close(animation._fig)
display(Image(filename='flash_progress.gif'))

## Optional GPU backend check

Enable this only in a compatible GPU runtime. The default notebook keeps GPU execution off. A float32 backend rejection is not evidence of a hardware incompatibility.

In [ ]:
RUN_GPU_CHECK = False
if RUN_GPU_CHECK and torch.cuda.is_available():
    for row in probe_attention_backends():
        print(row)

## Exercises and limits

Change block sizes and retain output and gradient checks. Add a causal mask and test fully masked rows. Measure backward-pass memory separately before making training-memory claims.

## References

Online softmax: https://arxiv.org/abs/1805.02867; Memory-efficient attention: https://arxiv.org/abs/2112.05682; FlashAttention-2: https://arxiv.org/abs/2307.08691